# 04. Phase 1 Results

Main architecture comparison for Gemini 2.5 Flash on the 300-question benchmark.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from notebooks.notebook_utils import *

set_plot_style()
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

phase1_raw, phase1_summary = load_phase1_results()
phase1_scored = phase1_raw[phase1_raw['faithfulness'].notna()].copy()
markdown_df(phase1_summary)

## Table 1-style summary

In [ ]:
table1 = markdown_df(phase1_summary[['architecture', 'queries', 'mean_faithfulness', 'mean_bertscore_f1', 'mean_hallucination_rate', 'zero_faithfulness_cases', 'mean_input_tokens', 'total_cost_usd', 'total_latency_s']])
table1

## Retrieval metrics for RAG conditions

In [ ]:
rag_only = phase1_summary[phase1_summary['architecture'].isin(['Simple RAG', 'Advanced RAG'])][['architecture', 'mean_context_precision', 'mean_context_recall']]
markdown_df(rag_only)

## Cost and latency comparison

In [ ]:
cost_table = phase1_summary[['architecture', 'mean_input_tokens', 'mean_total_cost_usd', 'total_cost_usd', 'total_latency_s']]
markdown_df(cost_table)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=phase1_summary, x='architecture', y='mean_faithfulness', ax=axes[0], palette='crest')
axes[0].set_title('Faithfulness')
sns.barplot(data=phase1_summary, x='architecture', y='mean_hallucination_rate', ax=axes[1], palette='flare')
axes[1].set_title('Hallucination rate')
sns.barplot(data=phase1_summary, x='architecture', y='mean_bertscore_f1', ax=axes[2], palette='mako')
axes[2].set_title('BERTScore F1')
for ax in axes:
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()

## By question type

In [ ]:
question_breakdown = phase1_scored.groupby(['architecture', 'question_type'])['faithfulness'].mean().reset_index()
question_breakdown.pivot(index='question_type', columns='architecture', values='faithfulness').round(3)

In [ ]:
sns.barplot(data=question_breakdown, x='question_type', y='faithfulness', hue='architecture')
plt.xticks(rotation=20)
plt.title('Phase 1 faithfulness by question type')
plt.tight_layout()

## Statistical tests and LC failure concentration

In [ ]:
phase1_tests = phase1_pairwise_tests()
markdown_df(phase1_tests)

In [ ]:
failure_stratum = phase1_failure_by_stratum()
markdown_df(failure_stratum)